<div style="background-color: steelblue; padding: 10px; border-radius: 5px;">
    <p style="margin: 10;"></p>
    <h1 style="text-align: center; margin: 0; font-weight: bold; color: white;">SWOT OMIP : Global estimation of spectrum</h1>
    <p style="margin: 10;"></p>
</div>



## 📦**Imports** 

In [1]:
%%time

##################################
#Imports

from datetime import datetime
import s3fs
import xarray as xr
import pyinterp
import numpy as np
import json
from watermark import watermark
import time
import platform
from shapely.geometry import shape, box
from shapely import geometry
import geopandas as gpd
from shapely import geometry
import DataPreprocessing_listv2 as dp

from widetrax import Spectra as sp
import geopandas as gpd

CPU times: user 1.68 s, sys: 500 ms, total: 2.18 s
Wall time: 6.33 s


## 🧮 **Required variables**

In [2]:



season="JAS"
prefix='GLO36V1'
s3_folder =f"s3://project-moi-swot-omip/{prefix}" # Do not write /!!!!!!
endpoint_url =  "https://minio.dive.edito.eu/"





## 🔍 **Check the S3 Endpoint**  

In [3]:

fs = s3fs.S3FileSystem(anon=True,endpoint_url=endpoint_url)
# List contents of the bucket
contents = fs.ls(s3_folder)
print("Bucket contents:")
for item in contents:
    print(item)

Bucket contents:
project-moi-swot-omip/GLO36V1/cycle_.keep
project-moi-swot-omip/GLO36V1/cycle_008
project-moi-swot-omip/GLO36V1/cycle_009
project-moi-swot-omip/GLO36V1/cycle_010
project-moi-swot-omip/GLO36V1/cycle_011
project-moi-swot-omip/GLO36V1/cycle_012
project-moi-swot-omip/GLO36V1/cycle_013
project-moi-swot-omip/GLO36V1/cycle_014
project-moi-swot-omip/GLO36V1/cycle_015
project-moi-swot-omip/GLO36V1/cycle_016
project-moi-swot-omip/GLO36V1/cycle_017
project-moi-swot-omip/GLO36V1/cycle_018
project-moi-swot-omip/GLO36V1/cycle_019
project-moi-swot-omip/GLO36V1/cycle_020
project-moi-swot-omip/GLO36V1/cycle_021
project-moi-swot-omip/GLO36V1/cycle_022
project-moi-swot-omip/GLO36V1/cycle_023
project-moi-swot-omip/GLO36V1/cycle_024
project-moi-swot-omip/GLO36V1/cycle_025
project-moi-swot-omip/GLO36V1/cycle_026


## 🔄 **Identify the cycle numbers within the specified time range** 

In [4]:
# Nothing new here

if season=="JFM":    
    start_date = "01012024" # "DDMMYYYY"
    end_date ="31032024"
elif season=="JAS":
    start_date = "01072024" # "DDMMYYYY"
    end_date ="30092024"

if season =="JFM":
    file_path = "https://minio.lab.dive.edito.eu/project-meom-ige/cycles_periods.csv" # works only for winter period
elif season =="JAS":
    file_path = "time_ranges.csv"  # for summer

matching_cycles = dp.get_matching_cycles(file_path, start_date, end_date)

def formater_numeros_concis(liste_numeros):
  return [str(numero).zfill(3) for numero in liste_numeros]
    
matching_cycles = formater_numeros_concis(matching_cycles)
matching_cycles

['017', '018', '019', '020', '021']

In [5]:
%%time
import save_spect_json2 as ssj

import importlib

importlib.reload(dp)

with open("mostly_ocean_boxes_filtered.geojson") as f:
    data = json.load(f)
    
bx=187
for feature in data["features"][187:]:
    poly = shape(feature["geometry"])
    specific_area = poly
    #lon_min, lat_min, lon_max, lat_max
    area = list(poly.bounds)
    area[0]=(area[0] + 360) % 360
    area[2]=(area[2] + 360) % 360
    area = tuple(area)
    print(area)
    lon_min, lat_min, lon_max, lat_max = area

    phase = 'science'
    GEOMETRIES_FILE = f'KaRIn_2kms_{phase}_geometries.geojson'
    swath_geometries = gpd.read_file(GEOMETRIES_FILE)
    swath_geometries['geometry'] = swath_geometries.geometry.apply(dp.normalize_longitude)
    swath_geometries.crs = 'EPSG:4326'  # Ensure WGS84
    matching_polys = swath_geometries[swath_geometries.intersects(specific_area)]
    pass_numbers = list(matching_polys['pass_number'])
    list_files=dp.find_listdata(matching_cycles, endpoint_url,s3_folder, pass_numbers, start_date, end_date)

    datasets_dict = dp.read_swot_ncfiles_S3folder_list(s3_folder, endpoint_url, area, list_files,engine="h5netcdf")

    
    has_converged, filled_datasets = dp.fill_nan(datasets_dict, varname = "ssh")

    segments_dict = sp.retrieve_segments(filled_datasets,FileType = "NetCDF",namevar="ssh")

    psd_dict, freqs_dict = sp.calculate_psd(segments_dict)
    # Calculate PSD Mean
    psd_mean, freqs_mean = sp.psd_mean_and_freq(psd_dict,freqs_dict)

    ssj.save_spect(("GLOBALv2/GLO36V1/Global_box_"+str(bx)),"GLO36V1",season,freqs_mean,psd_mean,area[0],area[2],area[1],area[3],start_date,end_date)
    bx=bx+1
# I should save for every region or just one file
# In the bucket I have to create a folder to save  outputs

(160.0, -10.0, 170.0, 0.0)
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_336_20240701T004948_20240701T014115_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_364_20240702T005019_20240702T014145_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_392_20240703T005050_20240703T014216_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_405_20240703T115939_20240703T125105_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_433_20240704T120009_20240704T125136_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_446_20240704T230858_20240705T000024_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_461_20240705T120040_20240705T125206_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_489_20240706T120111_20240706T125238_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_517_20240707T120142_20240707T

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(170.0, -10.0, 180.0, 0.0)
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_349_20240701T115837_20240701T125003_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_377_20240702T115908_20240702T125034_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_405_20240703T115939_20240703T125105_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_446_20240704T230858_20240705T000024_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_474_20240705T230929_20240706T000055_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_502_20240706T231000_20240707T000126_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_530_20240707T231031_20240708T000158_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_558_20240708T231102_20240709T000229_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_571_2024070

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(180.0, 0.0, 190.0, 10.0)
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_362_20240701T230726_20240701T235852_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_390_20240702T230756_20240702T235923_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_418_20240703T230827_20240703T235954_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_487_20240706T101818_20240706T110944_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_515_20240707T101849_20240707T111016_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_543_20240708T101920_20240708T111047_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_571_20240709T101951_20240709T111118_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_018/SWOT_GRID_L3_LR_SSH_018_015_20240710T102023_20240710T111149_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_018/SWOT_GRID_L3_LR_SSH_018_028_20240710

...

JSON file created
(340.0, 50.0, 350.0, 60.0)
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_337_20240701T014115_20240701T023241_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_348_20240701T110710_20240701T115836_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_365_20240702T014146_20240702T023312_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_376_20240702T110741_20240702T115907_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_393_20240703T014217_20240703T023343_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_404_20240703T110812_20240703T115939_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_421_20240704T014248_20240704T023414_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_432_20240704T110843_20240704T120009_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_433_2024070

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(170.0, 50.0, 180.0, 60.0)
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_362_20240701T230726_20240701T235852_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_390_20240702T230756_20240702T235923_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_418_20240703T230827_20240703T235954_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_433_20240704T120009_20240704T125136_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_461_20240705T120040_20240705T125206_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_489_20240706T120111_20240706T125238_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_517_20240707T120142_20240707T125309_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_528_20240707T212738_20240707T221904_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_017/SWOT_GRID_L3_LR_SSH_017_545_2024070

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
CPU times: user 30min 8s, sys: 24min 19s, total: 54min 28s
Wall time: 3h 55min 10s


## 💾 **Save Results and Information in JSON File**

## 📤 **Export Results to the S3 Endpoint** 

In [14]:
fs = s3fs.S3FileSystem( anon=True, endpoint_url="https://minio.lab.dive.edito.eu", use_ssl=False ) 
json_files = glob.glob(f"{model}/*.json")

for f in json_files:
    fs.put(f, f"project-meom-ige/OMIP/GLOBAL/{model}/{f.split('/')[-1]}")

[None]